# 🤖 Proto 1: Learned Controllers for Latent-Preserving Context Switching
### Complete Google Colab Runner (T4 GPU Accelerated)

Welcome! This notebook trains and evaluates **Proto 1** of our ANLP project.

### 🧸 Explain Like I'm 5 (What is this doing?):
- Imagine **Agent A** and **Agent B** are two friends sharing a secret topic (like a surprise birthday party).
- Suddenly, an outsider **Agent C** (an eavesdropper) walks into the room!
- **Agent A** needs to change the topic right away (for instance, start talking about gardening) so that **Agent C has no idea** about the surprise party.
- But **Agent A** must choose the new topic so cleverly that **friend Agent B still understands the secret message!**
- We train a tiny, super-smart brain called a **Controller** using Reinforcement Learning (REINFORCE) to pick the best cover topic every time.

---

## Step 1: Check GPU & Install Packages
Make sure your Colab runtime has a GPU enabled (**Runtime → Change runtime type → T4 GPU**).

In [ ]:
# Verify GPU
!nvidia-smi

# Install required libraries
!pip install -q transformers accelerate wandb huggingface_hub torch matplotlib

## Step 2: Clone Repository & Enter Directory
If you haven't cloned the GitHub repository yet, run the cell below to fetch the latest code.

In [ ]:
import os
if not os.path.exists("proto_1"):
    !git clone https://github.com/Chiranjeevi-2020/ANLP_grp_project.git
    %cd ANLP_grp_project
else:
    print("Already inside project folder!")

!ls proto_1/

## Step 3: Set up WandB & Hugging Face (Optional)
Enter your free API keys below to log your training live to **Weights & Biases (wandb)** and save controller checkpoints to **Hugging Face Hub**.

> 💡 **Tip:** If you do not have an account or key right now, simply leave the input blank and hit **Enter**! Everything will run smoothly and save locally.

In [ ]:
import os
from getpass import getpass

# Weights & Biases (wandb)
wandb_key = getpass("Enter WandB API Key (or press Enter to skip): ").strip()
if wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
    os.environ["WANDB_PROJECT"] = "anlp-latent-context-switching"
    print("✅ WandB online tracking enabled!")
else:
    os.environ["WANDB_MODE"] = "offline"
    print("ℹ️ WandB offline mode enabled (results saved locally).")

# Hugging Face Hub
hf_token = getpass("Enter Hugging Face Token (or press Enter to skip): ").strip()
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    repo = input("Enter HF Repo ID (optional, e.g. username/anlp-proto1) [press Enter for auto]: ").strip()
    if repo:
        os.environ["HF_REPO_ID"] = repo
    print("✅ Hugging Face Hub checkpoint upload enabled!")
else:
    print("ℹ️ Hugging Face upload skipped (checkpoints saved locally).")

## Step 4: Run Training & Evaluation Pipeline
We have provided pre-tuned configurations:
- `proto_1/configs/colab_fast.json` (~10-15 minutes on T4 GPU) ⭐ **Recommended for Colab**
- `proto_1/configs/full.json` (Full 2-hour paper benchmark)
- `proto_1/configs/tiny.json` (1-minute sanity test)

In [ ]:
# Run the full automated pipeline with live WandB logging
!python -m proto_1.run_colab --config proto_1/configs/colab_fast.json --no-input

## Step 5: Run Multi-Lambda Sweep (RQ3 Trade-off Curve)
This sweeps across penalty weights $\lambda \in \{0.0, 0.5, 1.0, 2.0\}$, computes insider-vs-outsider accuracy trade-offs, and plots the **Pareto frontier**.

In [ ]:
!python -m proto_1.sweep_lambda --config proto_1/configs/colab_fast.json --lambdas 0.0 0.5 1.0 2.0

## Step 6: Linguistic Naturalness & Diversity Audit
This checks that the switched dialogue is fluent, measures **Perplexity (PPL)** under the frozen LM, and calculates lexical richness (Distinct-1, Distinct-2).

In [ ]:
!python -m proto_1.linguistic_audit --config proto_1/configs/colab_fast.json

## Step 7: View Results & Visualizations Inline

In [ ]:
from IPython.display import display, Markdown, Image
import glob

# Display Results Markdown
results_files = glob.glob("results/proto_1/*/eval/RESULTS.md")
if results_files:
    with open(results_files[0]) as f:
        display(Markdown(f.read()))

# Display Training Curve
curve_files = glob.glob("results/proto_1/*/rl/training_curve.png")
if curve_files:
    print("Training Curve:")
    display(Image(curve_files[0]))

# Display Lambda Sweep Pareto Frontier
sweep_files = glob.glob("results/proto_1/sweep/lambda_tradeoff.png")
if sweep_files:
    print("Lambda Sweep Trade-off:")
    display(Image(sweep_files[0]))